# Lab 16: Vector and hybrid retrieval

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Project-Lantern-Case-Study-Lab2-Team-2 | LLM mode: replay


One chunk per block here (the sample is small). Scope is enforced inside the store and on the lexical side.

In [2]:
import numpy as np, shutil
from src import sample_lsa
recs = sd.records("deal_a") + sd.records("deal_b")
chunks = [{"chunk_id": f'{r["doc_id"]}:{r["block_id"]}', "deal_id": r["deal_id"], "doc_id": r["doc_id"], "tier": r["tier"],
           "block_ids": [r["block_id"]], "text": r["text"]} for r in recs]
vecs = llm.embed([c["text"] for c in chunks], sample_lsa.MODEL, compute=sample_lsa.encode).tolist()
embed = lambda text: llm.embed([text], sample_lsa.MODEL, compute=sample_lsa.encode)[0].tolist()
import chromadb                    # rerun-safe: drop the collection through the client, not the folder
try:
    chromadb.PersistentClient(path="data/index").delete_collection("lantern")
except Exception:
    pass                           # first run: nothing to delete

In [3]:
# From the slide: A vector store with scope enforced in the store
import chromadb, json

client = chromadb.PersistentClient(path="data/index")
col = client.get_or_create_collection(
    "lantern", configuration={"hnsw": {"space": "cosine"}})

col.add(ids=[c["chunk_id"] for c in chunks],
        embeddings=vecs,
        documents=[c["text"] for c in chunks],
        metadatas=[{"deal_id": c["deal_id"],
                    "doc_id": c["doc_id"],
                    "tier": c["tier"],
                    "block_ids": json.dumps(c["block_ids"])}
                   for c in chunks])

def search(qvec, scope: str, k: int = 10):
    if not scope:
        raise ValueError("scope is required")
    return col.query(query_embeddings=[qvec], n_results=k,
        where={"deal_id": {"$in": [scope, "public"]}})

In [4]:
# From the slide: BM25 plus dense, fused with reciprocal rank fusion
from rank_bm25 import BM25Okapi
tok = lambda s: s.lower().split()
bm25 = BM25Okapi([tok(c["text"]) for c in chunks])

def rrf(*rankings, k=60):
    score = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            score[doc] = (score.get(doc, 0)
                          + 1 / (k + rank))
    return sorted(score, key=score.get, reverse=True)

def hybrid(query, qvec, scope, k=10, rrf_k=60):
    dense = search(qvec, scope, k=50)["ids"][0]
    s = bm25.get_scores(tok(query))
    ok = (scope, "public")         # scope BM25 too
    lex = [chunks[i]["chunk_id"]
           for i in s.argsort()[::-1]
           if chunks[i]["deal_id"] in ok][:50]
    return rrf(dense, lex, k=rrf_k)[:k]

In [5]:
hits = hybrid("customer concentration", embed("customer concentration"), scope="deal_a", k=5)
print(hits)
assert all(not h.startswith("DR-deal_b") for h in hits)
try:
    search(embed("customer concentration"), scope="")
except ValueError as e:
    print("as expected:", e)

['DR-deal_a-001:p0003_b001', 'DR-deal_a-002:t002', 'DR-deal_a-003:p0001_b003', 'DR-deal_a-001:p0004_b002', 'DR-deal_a-001:p0002_b001']
as expected: scope is required


**Exercise 16 — compare RRF settings with retrieval evidence.**
Evaluate BM25, dense retrieval, and hybrid retrieval with `rrf_k=10` and `rrf_k=60` on the same 12 labelled queries. Report mean recall@5, MRR, and nDCG@10, then count queries whose ordered top-10 lists change. Here `k` limits returned results; `rrf_k` controls the reciprocal-rank denominator.

In [6]:
# From the slide: The metrics, implemented and tested
import math

def recall_at_k(ranked, rel, k):
    return len(set(ranked[:k]) & set(rel)) / len(rel)

def mrr(ranked, rel):
    for i, d in enumerate(ranked, 1):
        if d in rel:
            return 1 / i
    return 0.0

def ndcg_at_k(ranked, grades, k):
    gain = lambda g, i: (2**g - 1) / math.log2(i + 1)
    dcg = sum(gain(grades.get(d, 0), i)
              for i, d in enumerate(ranked[:k], 1))
    ideal = sorted(grades.values(), reverse=True)[:k]
    idcg = sum(gain(g, i)
               for i, g in enumerate(ideal, 1))
    return dcg / idcg if idcg else 0.0
# Small hand-checked example: the relevant result is ranked second.
assert recall_at_k(["a", "b"], ["b"], 1) == 0
assert recall_at_k(["a", "b"], ["b"], 2) == 1
assert mrr(["a", "b"], ["b"]) == 0.5
assert math.isclose(ndcg_at_k(["a", "b"], {"b": 2}, 2), 1 / math.log2(3))


In [7]:
import pandas as pd
qrels = [json.loads(line) for line in Path("data/eval/qrels.jsonl").read_text().splitlines() if line.strip()]
assert len(qrels) == 12
chunk_by_id = {c["chunk_id"]: c for c in chunks}
rows, comparisons = [], []
for q in qrels:
    qvec = embed(q["query"])
    dense = search(qvec, q["scope"], k=min(50, len(chunks)))["ids"][0]
    scores = bm25.get_scores(tok(q["query"]))
    lexical = [chunks[i]["chunk_id"] for i in scores.argsort()[::-1]
               if chunks[i]["deal_id"] in (q["scope"], "public")][:50]
    # Both fusion settings receive exactly the same candidate rankings.
    rankings = {
        "bm25": lexical,
        "dense": dense,
        "hybrid rrf_k=10": rrf(dense, lexical, k=10),
        "hybrid rrf_k=60": rrf(dense, lexical, k=60),
    }
    for rrf_k in (10, 60):
        assert hybrid(q["query"], qvec, q["scope"], k=10, rrf_k=rrf_k) == rankings[f"hybrid rrf_k={rrf_k}"][:10]
    for name, ranked in rankings.items():
        assert all(chunk_by_id[cid]["deal_id"] in (q["scope"], "public") for cid in ranked)
        rows.append({
            "qid": q["qid"], "retriever": name,
            "recall@5": recall_at_k(ranked, q["relevant"], 5),
            "mrr": mrr(ranked, q["relevant"]),
            "ndcg@10": ndcg_at_k(ranked, q["relevant"], 10),
        })
    a, b = rankings["hybrid rrf_k=10"], rankings["hybrid rrf_k=60"]
    comparisons.append({
        "qid": q["qid"], "query": q["query"],
        "top10_changed": a[:10] != b[:10],
        "top5_changed": a[:5] != b[:5],
        "full_ranking_changed": a != b,
        "top10_rrf10": a[:10], "top10_rrf60": b[:10],
    })
results = pd.DataFrame(rows)
summary = results.groupby("retriever")[["recall@5", "mrr", "ndcg@10"]].mean()
comparison = pd.DataFrame(comparisons)
display(summary.round(3))
display(comparison[["qid", "top10_changed", "top5_changed", "full_ranking_changed"]])
print(f"Top-10 lists changed for {comparison['top10_changed'].sum()} of {len(qrels)} queries.")
print("Mean metric changes (rrf_k=10 minus rrf_k=60):")
print((summary.loc["hybrid rrf_k=10"] - summary.loc["hybrid rrf_k=60"]).round(3))
print("Highest mean nDCG@10:", summary["ndcg@10"].idxmax())


,recall@5,mrr,ndcg@10
retriever,,,
bm25,0.875,0.873,0.850
dense,1.000,1.000,0.958
hybrid rrf_k=10,0.958,0.903,0.895
hybrid rrf_k=60,0.917,0.903,0.890


,qid,top10_changed,top5_changed,full_ranking_changed
0,w-01,False,False,True
1,w-02,True,True,True
2,w-03,True,False,True
3,w-04,False,False,False
4,w-05,False,False,True
5,w-06,True,True,True
6,w-07,False,False,True
7,w-08,False,False,True
8,w-09,False,False,True
9,w-10,False,False,False


Top-10 lists changed for 4 of 12 queries.
Mean metric changes (rrf_k=10 minus rrf_k=60):
recall@5    0.042
mrr         0.000
ndcg@10     0.005
dtype: float64
Highest mean nDCG@10: dense


**Interpretation.** A smaller `rrf_k` gives more weight to top-ranked candidates in each input list. The top-10 count above compares ordered lists, so reordering counts even if the set of retrieved blocks is unchanged. Full-ranking and top-five changes answer different questions and are reported separately.

Use the metric differences above to judge whether the ranking changes help: a positive difference favors `rrf_k=10`, a negative difference favors `rrf_k=60`, and zero is a tie. Dense retrieval has an artificial advantage on this classroom sample because `lsa-sample-v1` was fitted on this corpus. BM25 also uses a simple whitespace tokenizer. These 12 queries do not establish that dense retrieval or one RRF setting will win on a different corpus; repeat the evaluation on held-out queries for the real case study.

**Measured result in this environment:** 4 of 12 top-10 lists changed. RRF 10 improves mean recall@5 from 0.917 to 0.958 and nDCG@10 from 0.890 to 0.895; MRR stays 0.903. Dense retrieval scores highest (nDCG@10 0.958). These saved results differ from the handout reference (8 changed lists and slightly different nDCG values); they are the current run’s measurements, not values copied from the handout. Ranked-list differences, including the ordering of tied scores across library versions, can affect these comparisons.
